# 04 — Préparer les tables pour le Lakehouse Fabric

**Statut : préparatoire.** Ce notebook n'a pas encore été exécuté dans Fabric.

**Objectif** : transformer les données Citiz explorées, les contours administratifs et les paramètres de coût en **tables propres, typées et contrôlées**, prêtes pour un Lakehouse Microsoft Fabric.

**Lakehouse** : espace de stockage Fabric avec deux zones :
- `Files/` : des fichiers (CSV, Parquet, GeoJSON) ;
- `Tables/` : des tables **Delta**, interrogeables en SQL et lisibles par Power BI.

**Organisation proposée**

```
Files/raw/gbfs/        CSV bruts du notebook 01 (jamais modifiés)
Files/raw/boundaries/  contours régions / départements
Files/clean/           une copie Parquet de chaque table
Tables/                tables Delta
```

**Deux environnements**
- **Fabric** (`spark` existe) : lit `Files/raw/`, écrit `Files/clean/` puis les tables Delta.
- **Local** : écrit un miroir de cette arborescence dans `citiz/data/lakehouse/`, à déposer tel quel dans le Lakehouse.

**Règles appliquées à toutes les tables**
- noms de tables et de colonnes en anglais, minuscules, sans espace ni accent ;
- identifiants en texte ;
- dates en UTC à la microseconde (Spark ne lit pas les nanosecondes de pandas) ;
- géométries en texte WKT + `lat` / `lon` : Delta n'a pas de type géométrique ;
- une clé primaire par table, vérifiée avant écriture ;
- colonne `_loaded_at` : date du chargement.

In [1]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd

try:
    spark  # noqa: F821  (defined automatically in Fabric)
    IN_FABRIC = True
except NameError:
    IN_FABRIC = False

if IN_FABRIC:
    lakehouse_directory = Path("/lakehouse/default")
else:
    from pathlib import Path

    current_directory = Path.cwd().resolve()
    project_directory = None
    for directory in [current_directory, *current_directory.parents]:
        if (directory / "citiz" / "AGENTS.md").exists():
            project_directory = directory
            break
    if project_directory is None:
        raise FileNotFoundError("Open this notebook from the repository directory.")

    module_directory = project_directory / "citiz"
    gbfs_directory = module_directory / "data" / "raw" / "gbfs"
    boundaries_directory = module_directory / "data" / "raw" / "boundaries"
    processed_directory = module_directory / "data" / "processed"
    for directory in [gbfs_directory, boundaries_directory, processed_directory]:
        directory.mkdir(parents=True, exist_ok=True)
    lakehouse_directory = module_directory / "data" / "lakehouse"

raw_gbfs_directory = lakehouse_directory / "Files" / "raw" / "gbfs"
raw_boundaries_directory = lakehouse_directory / "Files" / "raw" / "boundaries"
clean_directory = lakehouse_directory / "Files" / "clean"
for directory in [raw_gbfs_directory, raw_boundaries_directory, clean_directory]:
    directory.mkdir(parents=True, exist_ok=True)

LOADED_AT = pd.Timestamp(datetime.now(timezone.utc))
print("Fabric" if IN_FABRIC else "Local", "->", lakehouse_directory if IN_FABRIC else lakehouse_directory.relative_to(project_directory))

Local -> citiz/data/lakehouse


## 1. Rassembler les fichiers bruts

- **En local** : copie des CSV de `citiz/data/raw/gbfs/` et des contours de `citiz/data/raw/boundaries/`.
- **Dans Fabric** : déposer d'abord les 4 CSV dans `Files/raw/gbfs/` (Lakehouse > Files > Upload), ou exécuter le notebook 01 dans Fabric. Les contours sont téléchargés s'ils manquent.

In [2]:
GBFS_FILES = ["citiz_flux.csv", "citiz_stations.csv", "citiz_station_status.csv", "citiz_vehicle_types.csv"]
BOUNDARY_URLS = {
    "regions": "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions-version-simplifiee.geojson",
    "departements": "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson",
}

if not IN_FABRIC:
    for file_name in GBFS_FILES:
        shutil.copy(gbfs_directory / file_name, raw_gbfs_directory / file_name)
    for name in BOUNDARY_URLS:
        local_copy = boundaries_directory / f"{name}.geojson"
        if local_copy.exists():
            shutil.copy(local_copy, raw_boundaries_directory / local_copy.name)

missing = [file_name for file_name in GBFS_FILES if not (raw_gbfs_directory / file_name).exists()]
assert not missing, f"Upload to Files/raw/gbfs/: {missing}"

for name, url in BOUNDARY_URLS.items():
    path = raw_boundaries_directory / f"{name}.geojson"
    if not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
print(sorted(path.name for path in raw_gbfs_directory.iterdir()))

['citiz_flux.csv', 'citiz_station_status.csv', 'citiz_stations.csv', 'citiz_vehicle_types.csv']


## 2. Outils communs

- `typed` : garde uniquement les colonnes du schéma, dans l'ordre, avec le bon type. Une colonne absente de la source est créée vide, pour que le schéma reste stable.
- `save_table` : vérifie la clé primaire, écrit le Parquet dans `Files/clean/`, puis la table Delta dans Fabric.

In [3]:
def typed(df, schema):
    """Project df on schema = {column: type}; types: string, Int64, float64, boolean, datetime."""
    result = pd.DataFrame(index=df.index)
    for column, dtype in schema.items():
        values = df[column] if column in df else pd.Series(pd.NA, index=df.index, dtype="object")
        if dtype == "datetime":
            result[column] = pd.to_datetime(values, utc=True, errors="coerce", format="ISO8601")
        else:
            result[column] = values.astype(dtype)
    return result.reset_index(drop=True)


def save_table(df, name, primary_key):
    """Check the primary key, then write Files/clean/<name>.parquet (and the Delta table in Fabric)."""
    duplicates = df.duplicated(primary_key).sum()
    assert duplicates == 0, f"{name}: {duplicates} duplicates on {primary_key}"
    assert df[primary_key].notna().all().all(), f"{name}: incomplete key {primary_key}"
    df = df.assign(_loaded_at=LOADED_AT)
    df.to_parquet(clean_directory / f"{name}.parquet", index=False,
                  coerce_timestamps="us", allow_truncated_timestamps=True)
    if IN_FABRIC:
        (spark.read.parquet(f"Files/clean/{name}.parquet")  # noqa: F821
              .write.format("delta").mode("overwrite")
              .option("overwriteSchema", "true").saveAsTable(name))
    print(f"{name:<30} {len(df):>6} rows  key={primary_key}")
    return df


ID_COLUMNS = {"station_id": "string", "vehicle_type_id": "string", "post_code": "string"}


def read_raw(file_name):
    return pd.read_csv(raw_gbfs_directory / file_name, dtype=ID_COLUMNS)


tables = {}

## 3. Référentiel géographique : `geo_region`, `geo_department`

Colonnes source renommées : `nom` → `name`. Le département reçoit le code de sa région (jointure spatiale sur un point intérieur), pour filtrer « Rhône » ou « Auvergne-Rhône-Alpes » sans recalcul géographique.

In [4]:
regions = gpd.read_file(raw_boundaries_directory / "regions.geojson")
departments = gpd.read_file(raw_boundaries_directory / "departements.geojson")

inner_points = departments.copy()
inner_points["geometry"] = departments.representative_point()
department_region = gpd.sjoin(inner_points, regions.rename(columns={"code": "region_code"})[["region_code", "geometry"]],
                              how="left", predicate="within")
departments["region_code"] = department_region["region_code"].values


def boundary_table(gdf, schema):
    df = pd.DataFrame(gdf.drop(columns="geometry")).rename(columns={"nom": "name"})
    df["geometry_wkt"] = gdf.geometry.to_wkt()
    centroids = gdf.to_crs("EPSG:2154").geometry.centroid.to_crs("EPSG:4326")
    df["centroid_lat"] = centroids.y
    df["centroid_lon"] = centroids.x
    return typed(df, schema)


BOUNDARY_SCHEMA = {"code": "string", "name": "string", "centroid_lat": "float64", "centroid_lon": "float64",
                   "geometry_wkt": "string"}
tables["geo_region"] = save_table(boundary_table(regions, BOUNDARY_SCHEMA), "geo_region", ["code"])
tables["geo_department"] = save_table(boundary_table(departments, {**BOUNDARY_SCHEMA, "region_code": "string"}),
                                      "geo_department", ["code"])

geo_region                         13 rows  key=['code']
geo_department                     96 rows  key=['code']


## 4. Citiz : réseaux, stations, types de véhicules

- `citiz_network` : un flux régional par ligne, clé `source`, avec un nom court (`network_name`).
- `citiz_station` : clé `station_uid`. L'état instantané (`num_vehicles_available`, `is_renting`) est retiré : il appartient à `citiz_station_status`. Les colonnes vides partout (`short_name`, `post_code`, `city`) sont gardées pour la stabilité du schéma.
- `citiz_vehicle_type` : clé `vehicle_type_uid` = `source:vehicle_type_id`. `max_range_meters` n'est pas reprise (0 partout, voir notebook 02).

In [5]:
feeds = read_raw("citiz_flux.csv")
feeds["network_name"] = feeds.dataset_title.str.replace("Autopartage Citiz ", "", regex=False)
tables["citiz_network"] = save_table(typed(feeds, {
    "source": "string", "network_name": "string", "dataset_title": "string", "publisher": "string",
    "gbfs_url": "string", "gbfs_version": "string", "feeds": "string", "licence": "string",
    "page_url": "string", "dataset_id": "string", "is_available": "boolean",
}), "citiz_network", ["source"])

citiz_network                      15 rows  key=['source']


In [6]:
stations = read_raw("citiz_stations.csv")
stations["station_uid"] = stations.source + ":" + stations.station_id
station_points = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.lon, stations.lat), crs="EPSG:4326")
station_points = gpd.sjoin(
    station_points,
    departments[["code", "region_code", "geometry"]].rename(columns={"code": "dep_code"}),
    how="left", predicate="within",
).drop(columns="index_right")
print(f"Stations without department: {station_points.dep_code.isna().sum()}")
station_points["geometry_wkt"] = station_points.geometry.to_wkt()

tables["citiz_station"] = save_table(typed(station_points, {
    "station_uid": "string", "source": "string", "station_id": "string", "name": "string",
    "address": "string", "post_code": "string", "city": "string",
    "lat": "float64", "lon": "float64", "geometry_wkt": "string",
    "dep_code": "string", "region_code": "string",
    "capacity": "Int64", "is_virtual_station": "boolean", "sources": "string",
    "extracted_at": "datetime",
}), "citiz_station", ["station_uid"])

Stations without department: 0
citiz_station                    1615 rows  key=['station_uid']


In [7]:
vehicle_types = read_raw("citiz_vehicle_types.csv")
vehicle_types["vehicle_type_uid"] = vehicle_types.source + ":" + vehicle_types.vehicle_type_id
tables["citiz_vehicle_type"] = save_table(typed(vehicle_types, {
    "vehicle_type_uid": "string", "source": "string", "vehicle_type_id": "string", "name": "string",
    "make": "string", "model": "string", "form_factor": "string", "propulsion_type": "string",
    "rider_capacity": "Int64",
}), "citiz_vehicle_type", ["vehicle_type_uid"])

citiz_vehicle_type                337 rows  key=['vehicle_type_uid']


## 5. Citiz : historique de disponibilité

- `citiz_station_status` : une ligne par station et par instantané (clé `station_uid` + `snapshot_ts`).
- `citiz_station_status_vehicle` : la colonne JSON `vehicle_types_available` dépliée, une ligne par type de véhicule disponible.

Rappel du notebook 02 : `last_reported` est souvent ancien. Ces tables sont conservées pour accumuler l'historique, mais leur usage comme mesure de fréquentation reste à valider.

In [8]:
status = read_raw("citiz_station_status.csv")
status["station_uid"] = status.source + ":" + status.station_id
status = status.drop_duplicates(["station_uid", "snapshot_ts"], keep="last")

tables["citiz_station_status"] = save_table(typed(status, {
    "station_uid": "string", "snapshot_ts": "datetime", "source": "string", "station_id": "string",
    "num_vehicles_available": "Int64", "num_vehicles_disabled": "Int64", "num_docks_available": "Int64",
    "is_installed": "boolean", "is_renting": "boolean", "is_returning": "boolean",
    "last_reported": "datetime",
}), "citiz_station_status", ["station_uid", "snapshot_ts"])

vehicle_rows = []
for row in status.itertuples():
    if isinstance(row.vehicle_types_available, str):
        for item in json.loads(row.vehicle_types_available):
            vehicle_rows.append({"station_uid": row.station_uid, "snapshot_ts": row.snapshot_ts,
                                 "vehicle_type_uid": f"{row.source}:{item['vehicle_type_id']}",
                                 "count": item.get("count")})
tables["citiz_station_status_vehicle"] = save_table(typed(pd.DataFrame(vehicle_rows), {
    "station_uid": "string", "snapshot_ts": "datetime", "vehicle_type_uid": "string", "count": "Int64",
}), "citiz_station_status_vehicle", ["station_uid", "snapshot_ts", "vehicle_type_uid"])

citiz_station_status             4844 rows  key=['station_uid', 'snapshot_ts']


citiz_station_status_vehicle     7543 rows  key=['station_uid', 'snapshot_ts', 'vehicle_type_uid']


## 6. Paramètres de coût : `cost_parameter`

Paramètres du futur modèle de coût (objectif 3). `status` vaut :
- `sourced` : chiffre tiré d'un document public ;
- `derived` : calculé à partir de chiffres sourcés ;
- `assumption` : hypothèse à fixer (valeur vide pour l'instant, à demander à Citiz).

Modifier cette cellule puis relancer pour mettre la table à jour.

In [9]:
CONVENTION_URL = "https://genevoisfrancais.org/wp-content/uploads/CS2021_23.2-AUTOPARTAGE_Convention-cadreAnnexe.pdf"
WHITE_PAPER_URL = "https://www.citiz.coop/uploads/2026/02/2026-CITIZ-livre_blanc.pdf"

costs = pd.DataFrame([
    ("citiz", "telematics_box", 4000, "EUR excl. VAT / car", "sourced", CONVENTION_URL + " (art. 5.3)"),
    ("citiz", "fixed_operating_cost", 250, "EUR / month / car", "sourced", CONVENTION_URL + " (art. 6.4)"),
    ("citiz", "revenue_per_km", 0.19, "EUR / km", "sourced", CONVENTION_URL + " (art. 6.4)"),
    ("citiz", "min_cars_per_station", 2, "cars", "sourced", CONVENTION_URL + " (preamble)"),
    ("citiz", "break_even_km", 250 / 0.19, "km / month / car", "derived", "fixed_operating_cost / revenue_per_km"),
    ("citiz", "car_purchase_price", None, "EUR incl. VAT / car", "assumption", "to ask Citiz"),
    ("citiz", "station_setup_cost", None, "EUR / station", "assumption", WHITE_PAPER_URL + " (described as minimal)"),
    ("bus_stop", "stop_creation_cost", None, "EUR / stop", "assumption", "to research"),
    ("bike_sharing", "station_creation_cost", None, "EUR / station", "assumption", "to research"),
], columns=["facility", "parameter", "value", "unit", "status", "source"])

tables["cost_parameter"] = save_table(typed(costs, {
    "facility": "string", "parameter": "string", "value": "float64",
    "unit": "string", "status": "string", "source": "string",
}), "cost_parameter", ["facility", "parameter"])

cost_parameter                      9 rows  key=['facility', 'parameter']


## 7. Contrôles de qualité

Vérifie les liens entre tables (clés étrangères) : chaque statut pointe vers une station connue, chaque station vers un réseau et un département connus, etc.

In [10]:
def check_foreign_key(child, column, parent, parent_column):
    orphans = (~tables[child][column].dropna().isin(tables[parent][parent_column])).sum()
    label = "OK" if orphans == 0 else "KO"
    print(f"{label}  {child}.{column} -> {parent}.{parent_column}: {orphans} orphans")
    return orphans


orphan_total = sum([
    check_foreign_key("citiz_station", "source", "citiz_network", "source"),
    check_foreign_key("citiz_station", "dep_code", "geo_department", "code"),
    check_foreign_key("citiz_station_status", "station_uid", "citiz_station", "station_uid"),
    check_foreign_key("citiz_station_status_vehicle", "vehicle_type_uid", "citiz_vehicle_type", "vehicle_type_uid"),
    check_foreign_key("citiz_vehicle_type", "source", "citiz_network", "source"),
    check_foreign_key("geo_department", "region_code", "geo_region", "code"),
])
assert orphan_total == 0

OK  citiz_station.source -> citiz_network.source: 0 orphans
OK  citiz_station.dep_code -> geo_department.code: 0 orphans
OK  citiz_station_status.station_uid -> citiz_station.station_uid: 0 orphans
OK  citiz_station_status_vehicle.vehicle_type_uid -> citiz_vehicle_type.vehicle_type_uid: 0 orphans
OK  citiz_vehicle_type.source -> citiz_network.source: 0 orphans
OK  geo_department.region_code -> geo_region.code: 0 orphans


In [11]:
data_dictionary = pd.DataFrame([
    {"table": name, "column": column, "dtype": str(df[column].dtype),
     "missing": int(df[column].isna().sum()), "rows": len(df)}
    for name, df in tables.items() for column in df.columns
])
data_dictionary.to_csv(clean_directory / "_data_dictionary.csv", index=False)
data_dictionary.groupby("table").agg(rows=("rows", "first"), columns=("column", "size"))

,rows,columns
table,,
citiz_network,15,12
citiz_station,1615,17
citiz_station_status,4844,12
citiz_station_status_vehicle,7543,5
citiz_vehicle_type,337,10
cost_parameter,9,7
geo_department,96,7
geo_region,13,6


## 8. Vérification dans Fabric

Requête SQL de contrôle (Fabric uniquement).

In [12]:
if IN_FABRIC:
    display(spark.sql("""
        SELECT d.name AS department, COUNT(*) AS stations, SUM(s.capacity) AS capacity
        FROM citiz_station s JOIN geo_department d ON s.dep_code = d.code
        GROUP BY d.name ORDER BY stations DESC LIMIT 10
    """))  # noqa: F821
else:
    print(f"Local files ready in {clean_directory.relative_to(project_directory)}")
    print("Upload citiz/data/lakehouse/Files/ to the Lakehouse Files/, then run this notebook in Fabric.")

Local files ready in citiz/data/lakehouse/Files/clean
Upload citiz/data/lakehouse/Files/ to the Lakehouse Files/, then run this notebook in Fabric.
